The below doesn't work, but at `options.json` right now is a correct result.

In [ ]:
import re
import json

# Define a function to extract options from the script
def extract_options(script):
    option_pattern = r'#\s*%option\s+%\s*key:\s*([\w]+)\s+%'
    matches = re.findall(option_pattern, script)
    return matches

# Read the script content
with open('opts_data.conf', 'r') as file:
    script_content = file.read()

# Extract all option keys from the script
options = extract_options(script_content)
print(options)

# Create a dictionary to store options and their details
option_dict = {}
for key in options:
    option_dict[key] = {
        'type': '',
        'description': '',
        'answer': ''
    }

# Update the dictionary with more detailed information if needed
# For now, we'll just use the keys to create a JSON structure

# Convert the dictionary to JSON
json_output = json.dumps(option_dict, indent=4)

# Save the JSON output to a file or print it
#with open('options.json', 'w') as json_file:
#    json_file.write(json_output)

print("JSON output has been saved to options.json")

[]
JSON output has been saved to options.json


In [3]:
import re
import json

def parse_conf_file(file_path):
    with open(file_path, 'r') as file:
        content = file.read()
    
    # Define regex patterns to extract relevant information
    option_pattern = r'#%option\s*(.*?)\s*#%END'
    flag_pattern = r'#%flag\s*(.*?)\s*#%end'
    
    options = re.findall(option_pattern, content)
    flags = re.findall(flag_pattern, content)
    
    # Combine options and flags into a single list for processing
    all_entries = options + flags
    
    schema = {}
    current_section = None
    
    for entry in all_entries:
        lines = entry.split('\n')
        
        key_line = next((line for line in lines if '#% key:' in line), '')
        type_line = next((line for line in lines if '#% type:' in line), '')
        gisprompt_line = next((line for line in lines if '#% gisprompt:' in line), '')
        description_line = next((line for line in lines if '#% description:' in line), '')
        answer_line = next((line for line in lines if '#% answer:' in line), '')
        section_line = next((line for line in lines if '#% guisection:' in line), '')
        
        key = key_line.split('#% key:')[1].strip() if key_line else None
        type_ = type_line.split('#% type:')[1].strip() if type_line else None
        gisprompt = gisprompt_line.split('#% gisprompt:')[1].strip() if gisprompt_line else None
        description = description_line.split('#% description:')[1].strip() if description_line else None
        answer = answer_line.split('#% answer:')[1].strip() if answer_line else None
        section = section_line.split('#% guisection:')[1].strip() if section_line else None
        
        # Handle the current section and add key to it
        if section:
            current_section = section
            if current_section not in schema:
                schema[current_section] = {}
        
        if key:
            entry_data = {
                "type": type_,
                "description": description,
                "answer": answer if answer else None
            }
            
            # If it's a flag, add the boolean type
            if entry.startswith('#%flag'):
                entry_data["type"] = "boolean"
            
            schema[current_section][key] = entry_data
    
    return json.dumps(schema, indent=4)

# Example usage
conf_file_path = 'agropastsemi_raw.conf'
json_schema = parse_conf_file(conf_file_path)
print(json_schema)

{}


In [1]:
import os

def parse_demo_conf(file_path):
    with open(file_path, 'r') as file:
        content = file.readlines()

    # Remove leading and trailing whitespace from each line
    content = [line.strip() for line in content]

    # Initialize the main dictionary to store all parsed data
    config_data = {}

    # Variables to keep track of current section and key-value pairs within that section
    current_section = None
    current_entry = {}
    
    # Iterate over each line in the file
    for line in content:
        if line.startswith('#% guisection:'):
            # Start a new section
            current_section = line.split(':')[1].strip()
            config_data[current_section] = []
        elif line.startswith('#%option') or line.startswith('#%flag'):
            # If we encounter an option, finalize the previous one and start a new entry
            if current_entry:
                config_data[current_section].append(current_entry)
            current_entry = {}
        else:
            # Parse key-value pairs from other lines
            parts = line.split('=')
            if len(parts) == 2:
                key, value = parts
                # Remove leading and trailing whitespace from both key and value
                key = key.strip()
                value = value.strip()
                
                # If the line is a description, strip any surrounding quotes
                if key.startswith('description'):
                    value = value.strip('"')
                
                # If the line is an answer with a list (like multiple=yes), split it
                if key == 'answer' and current_entry.get('multiple', False):
                    value = value.split(',')
                
                # Store the key-value pair in the current entry dictionary
                current_entry[key] = value

    # Append the last parsed entry to the current section (if any)
    if current_section and current_entry:
        config_data[current_section].append(current_entry)

    return config_data

# Path to the demo.conf file
demo_conf_path = 'demo.conf'

# Parse the configuration file
parsed_config = parse_demo_conf(demo_conf_path)

# Convert the parsed configuration to JSON format
import json
config_json = json.dumps(parsed_config, indent=4)

# Save the JSON output to a file (if needed)
with open('config_output.json', 'w') as json_file:
    json_file.write(config_json)

print("Configuration has been successfully parsed and saved to '/mnt/uploads/config_output.json'.")

Configuration has been successfully parsed and saved to '/mnt/uploads/config_output.json'.


In [4]:
import json
import re
from collections import defaultdict

def parse_text_to_nested_json(input_text, output_json_path):
    """
    Converts a configuration text (with sections and #%option blocks) into a nested JSON.
    
    :param input_text: String containing the full input text (or read from file)
    :param output_json_path: Path to save the resulting JSON file
    """
    # Split the text into lines
    lines = input_text.strip().splitlines()
    
    # Store final result
    result = {}
    current_section = None
    current_option = None
    option_stack = []  # Stack to handle multi-line values if needed (not used here)

    for line in lines:
        line = line.strip()
        
        # Detect section header: lines between ####... containing text
        if line.startswith("###################"):
            continue  # Skip the delimiter lines
        if line.startswith("#") and not line.startswith("#%"):
            # This is a candidate for a section header
            header_match = re.match(r"#\s*(.+)", line)
            if header_match:
                current_section = header_match.group(1).strip()
                result[current_section] = []
            continue
        
        # Handle #%option blocks
        if line == "#%option":
            current_option = {}
            continue
        
        if line == "#%END" and current_option is not None and current_section:
            result[current_section].append(current_option)
            current_option = None
            continue
        
        # Parse key-value pairs like: #% key: value
        if line.startswith("#% ") and current_option is not None:
            # Split only on the first colon
            part = line[3:]  # Remove '#% '
            if ':' in part:
                key, value = part.split(':', 1)
                key = key.strip()
                value = value.strip()
                
                # Type conversion where possible
                if value.lower() == 'yes':
                    value = True
                elif value.lower() == 'no':
                    value = False
                elif re.fullmatch(r'-?\d+\.?\d*', value):  # Simple number detection
                    value = float(value) if '.' in value else int(value)
                elif ',' in value and all(v.strip().lstrip('-').replace('.','').isdigit() for v in value.split(',')):
                    # Handle comma-separated numbers (e.g., "3,2")
                    try:
                        value = [float(v.strip()) if '.' in v else int(v.strip()) for v in value.split(',')]
                    except:
                        pass  # fallback to string

                current_option[key] = value

    # Write to JSON file
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=4, ensure_ascii=False)
    
    print(f"Nested JSON saved to {output_json_path}")

# Example usage:
if __name__ == "__main__":
    # Read input text (replace with file reading if needed)
    input_data = """
##################################
#Agent Properties
##################################
#%option
#% key: numpeople
#% type: double
#% description: Number of people in the village(s) (stays constant throughout the simulation)
#% answer: 30
#% guisection: Agent Properties
#%END
#%option
#% key: agentmem
#% type: integer
#% description: Length of the "memory" of the agent in years. The agent will use the mean surplus/defict information from this many of the most recent previous years when making a subsistence plan for the current year.
#% answer: 5
#% guisection: Agent Properties
#%END
#%option
#% key: aglabor
#% type: double
#% description: The amount of agricultural labor an average person of the village can do in a year (in "person-days")
#% answer: 300
#% guisection: Agent Properties
#%END



##################################
#Farming Options
##################################
#%option
#% key: agcatch
#% type: string
#% gisprompt: old,cell,raster
#% description: Map of the largest possible agricultural catchment map (From r.catchment or r.buffer, where catchment is a single integer value, and all else is NULL)
#% guisection: Farming
#%END
#%option
#% key: ewfieldsize
#% type: double
#% description: East-West dimension of fields in map units (Large field sizes may lead to some overshoot of catchment boundaries)
#% answer: 50
#% guisection: Farming
#%END
#%option
#% key: fieldlabor
#% type: double
#% description: Number of person-days required to till, sow, weed, and harvest one farm field in a year.
#% answer: 50
#% guisection: Farming
#%END
#%option
#% key: farmval
#% type: double
#% description: The landcover value for farmed fields (Should correspond to an appropriate value from your landcover rules file.)
#% answer: 5
#% guisection: Farming
#%END
#%option
#% key: farmimpact
#% type: double
#% description: The mean and standard deviation of the amount to which farming a patch decreases its fertility (in percentage points of maximum fertility, entered as: "mean,std_dev"). Fertility impact values of indvidual farm plots will be randomly chosen from a gaussian distribution that has this mean and std_dev.
#% answer: 3,2
#% multiple: yes
#% guisection: Farming
#%END
"""

#parse_text_to_nested_json(input_data, "output_config_brave.json")   

with open("agropastsemi_raw.conf", "r") as file:
    input_data = file.read()

parse_text_to_nested_json(input_data, "brave2.json")

Nested JSON saved to brave2.json
